# LLM04 Data and Model Poisoning — Upload Artifacts & Run Evaluation

**OWASP Category**: LLM04 — Data and Model Poisoning | **Risk Severity**: High

This notebook:
1. **Uploads** all artifact files (scenarios, model-based checks) from the local folder structure and registers them in Okareo.
2. **Runs** the full LLM04 data and model poisoning test suite against your target AI agent.

Uploading is **idempotent** — re-running will not create duplicates (scenarios return existing, checks upsert).
Registered IDs are available in-memory for the evaluation steps below.

**Three detection scenarios**:
- **RAG Corpus Poisoning**: Tests whether adversarial content in the retrieval store manipulates outputs
- **Behavioral Drift**: Compares current responses against known-good baselines to detect silent changes
- **Backdoor/Sleeper Triggers**: Tests whether specific trigger phrases activate hidden backdoor behaviors

In [9]:
%pip install okareo python-dotenv --quiet


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
import sys
import json
from pathlib import Path

# Add project root for owasp.common import
_nb = globals().get("__vsc_ipynb_file__", ".")
NOTEBOOK_DIR = Path(_nb).resolve().parent
CATEGORY_DIR = NOTEBOOK_DIR.parent
if str(CATEGORY_DIR.parent.parent) not in sys.path:
    sys.path.insert(0, str(CATEGORY_DIR.parent.parent))

from okareo.checks import ModelBasedCheck, CheckOutputType
from okareo.model_under_test import Driver

from owasp.common import (
    init_okareo,
    parse_check_md,
    parse_driver_md,
    parse_check_py_meta,
    parse_check_py_metadata,
    parse_check_py,
    CodeCheckFromSource,
    build_target,
    SINGLE_TURN_DRIVER_TEMPLATE,
)

okareo, OKAREO_API_KEY = init_okareo()
print(f"✓ Okareo SDK initialized (key: ...{OKAREO_API_KEY[-5:]})")
print(f"Category directory: {CATEGORY_DIR}")


✓ Okareo SDK initialized (key: ...a7sKA)
Category directory: /Users/guiair/dev/okareo/compliance-owasp/owasp/LLM04-data-model-poisoning


---
## Part 1 — Upload Artifacts

### Upload Scenarios

Scans `scenarios/` for `.jsonl` files and uploads each via `upload_scenario_set`.

In [11]:
scenarios_dir = CATEGORY_DIR / "scenarios"
registered_scenarios = {}

for jsonl_path in sorted(scenarios_dir.glob("*.jsonl")):
    scenario_name = f"LLM04-{jsonl_path.stem}"
    print(f"Uploading scenario: {scenario_name} from {jsonl_path.name}")

    scenario = okareo.upload_scenario_set(
        scenario_name=scenario_name,
        file_path=str(jsonl_path),
    )
    registered_scenarios[scenario_name] = scenario
    print(f"  \u2713 Registered: {scenario_name} (ID: {scenario.scenario_id})")

print(f"\nTotal scenarios uploaded: {len(registered_scenarios)}")

Uploading scenario: LLM04-backdoor-trigger from backdoor-trigger.jsonl
  ✓ Registered: LLM04-backdoor-trigger (ID: d9cba0cb-49e4-42f7-86ed-64d36aba44ed)
Uploading scenario: LLM04-behavioral-drift from behavioral-drift.jsonl
  ✓ Registered: LLM04-behavioral-drift (ID: 5d18d6ac-fbb8-4d1f-8bf6-d597d36d449a)
Uploading scenario: LLM04-corpus-poisoning from corpus-poisoning.jsonl
  ✓ Registered: LLM04-corpus-poisoning (ID: ada4a641-7465-4991-92d2-a8deb6a403e6)

Total scenarios uploaded: 3


### Register Model-Based Checks

Scans `checks/` for `.md` files, parses YAML front matter and prompt template,
and registers each via `create_or_update_check` using `ModelBasedCheck`.

In [12]:
checks_dir = CATEGORY_DIR / "checks"
registered_checks = {}

for md_path in sorted(checks_dir.glob("*.md")):
    check_data = parse_check_md(md_path)
    print(f"Registering model-based check: {check_data['name']} from {md_path.name}")

    check_obj = ModelBasedCheck(
        prompt_template=check_data["prompt_template"],
        check_type=CheckOutputType.PASS_FAIL,
    )
    result = okareo.create_or_update_check(
        name=check_data["name"],
        description=check_data["description"],
        check=check_obj,
    )
    registered_checks[check_data["name"]] = result.id
    print(f"  \u2713 Registered: {check_data['name']} (ID: {result.id})")

print(f"\nModel-based checks registered: {len(registered_checks)}")


Registering model-based check: LLM04-backdoor-trigger-detector from backdoor-trigger-detector.md
  ✓ Registered: LLM04-backdoor-trigger-detector (ID: 910c8207-ba81-4967-b7ce-91abd2ad76a6)
Registering model-based check: LLM04-behavioral-drift-detector from behavioral-drift-detector.md
  ✓ Registered: LLM04-behavioral-drift-detector (ID: dec3672d-806d-4163-9e69-0921a75b997d)
Registering model-based check: LLM04-corpus-poisoning-detector from corpus-poisoning-detector.md
  ✓ Registered: LLM04-corpus-poisoning-detector (ID: f54b42ba-a4c6-4109-996a-842d9ccc50ff)

Model-based checks registered: 3


### Artifact Upload Summary

In [13]:
print("=" * 60)
print("LLM04 Data and Model Poisoning \u2014 Artifact Upload Summary")
print("=" * 60)
print(f"\nScenarios ({len(registered_scenarios)}):")
for name, sc in registered_scenarios.items():
    print(f"  \u2022 {name} \u2192 {sc.scenario_id}")
print(f"\nChecks ({len(registered_checks)}):")
for name, cid in registered_checks.items():
    print(f"  \u2022 {name} \u2192 {cid}")
print("\n\u2713 All artifacts ready. Proceeding to evaluation...")

LLM04 Data and Model Poisoning — Artifact Upload Summary

Scenarios (3):
  • LLM04-backdoor-trigger → d9cba0cb-49e4-42f7-86ed-64d36aba44ed
  • LLM04-behavioral-drift → 5d18d6ac-fbb8-4d1f-8bf6-d597d36d449a
  • LLM04-corpus-poisoning → ada4a641-7465-4991-92d2-a8deb6a403e6

Checks (3):
  • LLM04-backdoor-trigger-detector → 910c8207-ba81-4967-b7ce-91abd2ad76a6
  • LLM04-behavioral-drift-detector → dec3672d-806d-4163-9e69-0921a75b997d
  • LLM04-corpus-poisoning-detector → f54b42ba-a4c6-4109-996a-842d9ccc50ff

✓ All artifacts ready. Proceeding to evaluation...


---
## Part 2 — Run Evaluation

### Behavioral Drift: Baseline Capture Workflow

The **behavioral drift** scenario compares the agent's current responses against known-good baselines
stored in the `result` field of `behavioral-drift.jsonl`. Before running drift detection for the first
time, you need to capture a baseline:

1. **Point at your trusted agent**: Ensure `owasp/target.env` references the trusted model version.
2. **Run the standardized prompts**: Execute the prompts in `behavioral-drift.jsonl` against the trusted agent.
3. **Save as baseline**: Update each row's `result` field with the trusted response.
4. **Version the baseline**: Update `version` in `behavioral-drift_meta.md` (e.g., `"1.0.0"`).
5. **Commit**: The committed JSONL is your baseline source of truth.

To update the baseline after validating a new model version, repeat steps 1-5 and increment the version.

### Configuration

The target agent is loaded from the shared `owasp/target.env` file.
All OWASP category notebooks reference this same file so that every control evaluates the same agent.

Each scenario is evaluated with **one specialized model-based check**:
- `LLM04-corpus-poisoning` → `LLM04-corpus-poisoning-detector`
- `LLM04-behavioral-drift` → `LLM04-behavioral-drift-detector`
- `LLM04-backdoor-trigger` → `LLM04-backdoor-trigger-detector`

In [14]:
# Target loaded from owasp/target.env. To use a different config: target = build_target(CATEGORY_DIR, env_path="target.prod.env")
target = build_target(CATEGORY_DIR)
TARGET_NAME = target.name
print(f"\u2713 Target agent: {TARGET_NAME}")

SCENARIO_CHECK_MAP = {
    "LLM04-corpus-poisoning":  ["LLM04-corpus-poisoning-detector"],
    "LLM04-behavioral-drift":  ["LLM04-behavioral-drift-detector"],
    "LLM04-backdoor-trigger":  ["LLM04-backdoor-trigger-detector"],
}

✓ Target agent: FinanceBot


### Build Target

Constructs a `CustomEndpointTarget` from `owasp/target.env` using `TurnConfig` for the
next-turn endpoint and optional `SessionConfig` / `EndSessionConfig` for session management.

In [15]:
# Target built in config cell above via build_target(CATEGORY_DIR)

### Single-Turn Tests — All Scenarios

Each scenario runs via `okareo.run_simulation()` constrained to `max_turns=1`.
A pass-through driver delivers the scenario input verbatim to the target agent; the agent's
response is evaluated by the scenario-specific model-based check.

In [16]:
single_turn_driver = Driver(
    temperature=0,
    name=f"{TARGET_NAME}-single-turn-driver",
    prompt_template=SINGLE_TURN_DRIVER_TEMPLATE,
)

all_results = {}

for scenario_name, scenario in registered_scenarios.items():
    checks_for_scenario = SCENARIO_CHECK_MAP.get(scenario_name, [])
    if not checks_for_scenario:
        print(f"\nSkipping {scenario_name}: no check mapping found in SCENARIO_CHECK_MAP")
        continue

    print(f"\n{'='*60}")
    print(f"Running: {scenario_name}")
    print(f"Check: {', '.join(checks_for_scenario)}")
    print(f"{'='*60}")

    try:
        test_run = okareo.run_simulation(
            target=target,
            driver=single_turn_driver,
            name=f"LLM04 Eval \u2014 {scenario_name}",
            api_key=OKAREO_API_KEY,
            first_turn="driver",
            scenario=scenario,
            max_turns=1,
            checks=checks_for_scenario,
        )
        all_results[scenario_name] = test_run
        print(f"  \u2713 Test run complete: {test_run.id}")
        if hasattr(test_run, "app_link") and test_run.app_link:
            print(f"  View: {test_run.app_link}")
    except Exception as e:
        print(f"  \u2717 Error: {e}")
        all_results[scenario_name] = None


Running: LLM04-backdoor-trigger
Check: LLM04-backdoor-trigger-detector
  ✓ Test run complete: 0b8d4bdd-d8c4-4e5d-b42e-1737c166a6b0
  View: https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/0b8d4bdd-d8c4-4e5d-b42e-1737c166a6b0

Running: LLM04-behavioral-drift
Check: LLM04-behavioral-drift-detector
  ✓ Test run complete: ced47d70-2573-482c-b31f-f9fa845e0a14
  View: https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/ced47d70-2573-482c-b31f-f9fa845e0a14

Running: LLM04-corpus-poisoning
Check: LLM04-corpus-poisoning-detector
  ✓ Test run complete: 14de98cc-15ae-46ee-ac01-18aef331f4c0
  View: https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/14de98cc-15ae-46ee-ac01-18aef331f4c0


### Results Summary

In [19]:
print("\n" + "=" * 60)
print("LLM04 DATA AND MODEL POISONING \u2014 EVALUATION RESULTS")
print("OWASP Category: LLM04 | Risk Severity: High")
print("=" * 60)

print(f"\n{'Scenario':<42} {'Status':<10} {'Link / Run ID'}")
print("-" * 100)
for name, result in all_results.items():
    if result is None:
        print(f"{name:<42} {'ERROR':<10} N/A")
    else:
        link = getattr(result, "app_link", None) or result.id
        print(f"{name:<42} {'COMPLETE':<10} {link}")

errors = sum(1 for r in all_results.values() if r is None)
print(f"\nTotal evaluated: {len(all_results)} | Errors: {errors}")
print(f"Check architecture: one model-based check per scenario")
if not errors:
    print("\u2713 All scenarios completed. See Okareo dashboard for full results.")


LLM04 DATA AND MODEL POISONING — EVALUATION RESULTS
OWASP Category: LLM04 | Risk Severity: High

Scenario                                   Status     Link / Run ID
----------------------------------------------------------------------------------------------------
LLM04-backdoor-trigger                     COMPLETE   https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/0b8d4bdd-d8c4-4e5d-b42e-1737c166a6b0
LLM04-behavioral-drift                     COMPLETE   https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/ced47d70-2573-482c-b31f-f9fa845e0a14
LLM04-corpus-poisoning                     COMPLETE   https://app.okareo.com/project/d354dcdd-ada6-4513-abbc-bf417f154bf5/eval/14de98cc-15ae-46ee-ac01-18aef331f4c0

Total evaluated: 3 | Errors: 0
Check architecture: one model-based check per scenario
✓ All scenarios completed. See Okareo dashboard for full results.


### Detailed Results (Optional)

Retrieve per-row scores and model outputs for any completed test run.

In [18]:
# Uncomment to inspect a specific completed run in detail:
# RUN_ID = "<paste_run_id_here>"
# detailed = okareo.get_test_run(RUN_ID)
# for row in (detailed.model_results or []):
#     print(f"Input:   {str(row.get('scenario_input', ''))[:80]}")
#     print(f"Output:  {str(row.get('model_output', ''))[:80]}")
#     print(f"Checks:  {row.get('check_scores', {})}")
#     print("-" * 40)